In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [2]:
import math
import random
import matplotlib.pyplot as plt
import json

### Task 1: Loading Dataset

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

Device : cuda


#### Task 1.1: Dataset Ingestion [2 Marks]

In [8]:
def load_data(file_path):
    eng, ger = [], []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                eng.append(data['en'])
                ger.append(data['de'])
    return eng, ger

In [9]:
train_eng, train_ger = load_data('train.jsonl')
val_eng, val_ger = load_data('val.jsonl')
test_eng, test_ger = load_data('test.jsonl')

#### Task 1.2: Sanity Inspection

In [10]:
print(f"Train Dataset size : {len(train_eng)}")
print(f"Validation Dataset size : {len(val_eng)}")
print(f"Test dataset size : {len(test_eng)}")
print(f"sample sentence - English: {train_eng[67]} | German: {train_ger[67]}")

Train Dataset size : 29000
Validation Dataset size : 1014
Test dataset size : 1000
sample sentence - English: A man and a little girl happily posing in front of their cart in a supermarket. | German: Ein Mann und ein kleines Mädchen posieren glücklich vor ihrem Einkaufswagen im Supermarkt.


### Task 2: Preprocessing and Data Pipeline

#### Task 2.1: Preprocessing & Vocabulary Construction

In [6]:
def tokenize(text):
    return text.lower().split()

In [11]:
MAX_LEN = 32
pad_idx, sos_idx, eos_idx, unk_idx = 0,1,2,3

In [12]:
def generate_vocab(sentences):
    token2idx = {"<pad>" : pad_idx, "<sos>" : sos_idx, "<eos>" : eos_idx, "<unk>" : unk_idx}
    idx2token = {pad_idx : "<pad>", sos_idx : "<sos>", eos_idx : "<eos>", unk_idx : "<unk>"}
    i = 4
    for s in sentences:
        for token in tokenize(s):
            if token not in token2idx:
                token2idx[token] = i
                idx2token[i] = token
                i += 1
    return token2idx, idx2token

In [13]:
source_t2i, source_i2t = generate_vocab(train_eng)
target_t2i, target_i2t = generate_vocab(train_ger)

In [14]:
def encode_sentence(sentence, token2idx, max_len=MAX_LEN):
    tokens = tokenize(sentence)
    tokens = tokens[:max_len]
    ids = [sos_idx] + [token2idx.get(tok, unk_idx) for tok in tokens] + [eos_idx]
    if len(ids) < max_len:
        ids += [pad_idx] * (max_len - len(ids))
    return ids

#### Task 2.2: Batch Data Generation

In [17]:
class DataGenerator(Dataset):
    def __init__(self, source_sentences, source_vocab, target_sentences, target_vocab):
        self.source_data = [encode_sentence(s, source_vocab) for s in source_sentences]
        self.target_data = [encode_sentence(t, target_vocab) for t in target_sentences]
    def __len__(self):
        return len(self.source_data)
    def __getitem__(self,index):
        return (torch.tensor(self.source_data[index], dtype=torch.long)), (torch.tensor(self.target_data[index], dtype=torch.long))

In [16]:
def get_batches(source_sent, target_sent, source_vocab, target_vocab, batch_size, shuffle=True):
    dataset = DataGenerator(source_sent, source_vocab, target_sent, target_vocab)
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    for source_batch, target_batch in dl:
        yield source_batch.to(device), target_batch.to(device)

### Task 3: Transformer Model Architecture

#### Task 3.1: Positional Encoding & Feed-Forward Sublayer

In [18]:
class sinPE(nn.Module):
    def __init__(self,d_model, max_len=MAX_LEN):
        super.__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [19]:
class posFFN(nn.Module):
    def __init__(self, d_model, d_ff):
        self.fcl1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.fcl2 = nn.Linear(d_ff, d_model)
    def forward(self, x):
        return self.fcl2(self.relu(self.fcl1(x)))

#### Task 3.2: Encoder & Decoder Layers

In [21]:
class encoder_layer(nn.Module):
    def __init__(self, d_model=256, head=512, d_ff=512, dropout=0.1):
        super.__init__()
        self.self_attn = nn.MultiheadAttention(d_model,head, dropout=dropout, batch_first=True)
        self.ffn = posFFN(d_model, d_ff)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, source, source_msk = None, sorce_key_padding_mask=None):
        attn_opt, _ = self.self_attn(source, source, source, key_padding_mask = sorce_key_padding_mask, attn_mask = source_msk)
        source = self.ln1(source + self.dropout(attn_opt))
        ffn_opt = self.ffn(source)
        source = self.ln2(source + self.dropout(ffn_opt))
        return source

In [22]:
class decoder_layer(nn.Module):
    def __init__(self, d_model=256, head=512, d_ff=512, dropout=0.1):
        super.__init__()
        self.self_attn = nn.MultiheadAttention(d_model, head, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, head, dropout=dropout, batch_first=True)
        self.ffn = posFFN(d_model, d_ff)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, target, memory, target_msk=None, target_key_padding_mask=None, memory_key_padding_mask=None):
        attn_opt, _ = self.self_attn(target, target, target, key_padding_mask = target_key_padding_mask, attn_mask = target_msk)
        target = self.ln1(target + self.dropout(attn_opt))
        cross_attn_opt, _ = self.cross_attn(target, memory, memory, key_padding_mask = memory_key_padding_mask)
        target = self.ln2(target + self.dropout(cross_attn_opt))
        ffn_opt = self.ffn(target)
        target = self.ln3(target + self.dropout(ffn_opt))
        return target

#### Task 3.3: Attention Masking Mechanisms

In [ ]:
def make_key_padding_mask(sequence, padding=pad_idx):
    return (sequence == padding)

In [24]:
def generate_causal_mask(size, device):
    mask = torch.triu(torch.ones(size, size, device=device) == 1, diagonal=1)
    return mask

#### Task 3.4: Complete Transformer Integration

In [25]:
class FullTransformer(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size, d_model=256, nhead=4, num_layers=3, d_ff=512, dropout=0.1):
        super.__init__()
        self.source_embed = nn.Embedding(source_vocab_size, d_model, padding_idx=pad_idx)
        self.target_embed = nn.Embedding(target_vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc = sinPE(d_model)
        self.enc_layers = nn.ModuleList([encoder_layer(d_model, nhead, d_ff, dropout) for _ in range(num_layers)])
        self.dec_layers = nn.ModuleList([decoder_layer(d_model, nhead, d_ff, dropout) for _ in range(num_layers)])
        self.fc_out = nn.Linear(d_model, target_vocab_size)
        self.d_model = d_model

    def encode(self, source, source_key_padding_mask):
        x = self.pos_enc(self.source_embed(source) * math.sqrt(self.d_model))
        for layer in self.enc_layers:
            x = layer(x, source_key_padding_mask = source_key_padding_mask)
        return x

    def decode(self, target, target_key_padding_mask):
        x = self.pos_enc(self.target_embed(target) * math.sqrt(self.d_model))
        for layer in self.dec_layers:
            x = layer(x, target_key_padding_mask=target_key_padding_mask)
        return self.fc_out(x)

    def forward(self, source, target):
        source_key_padding_mask = make_key_padding_mask(source)
        target_key_padding_mask = make_key_padding_mask(target)
        target_seq_len = target.size()
        target_mask = generate_causal_mask(target_seq_len, target.device)
        memory = self.encode(source, source_key_padding_mask)
        out = self.decode(target,memory, target_mask, target_key_padding_mask)
        return out